# 02 - Preprocessing (Stage 2)
**Owner:** Aiman | **Calendar:** Week 10 (19-25 Oct) | **Feeds:** Amna's baseline run (Week 10 prototype)

What this notebook does:
1. Loads the labels of EyePACS (Drive) and APTOS (local)
2. Crop the black border, pad to a square, resize to **512**, **CLAHE**, circular mask
3. **Patient-aware split 70/15/15** (frozen: one split file for the whole team)
4. Saves `manifest.csv`, one zip per dataset and the parameters on the shared Drive
5. Checks everything, then (optional) cleans up raw data

**How to run**
- First run with `MODE = "smoke"` (about 5 minutes, writes nothing to Drive).
- If the smoke run is clean, change to `MODE = "full"` and run all cells again (about 1 hour).

**What we do NOT do here:** no augmentation, no SMOTE, no oversampling. The files keep the natural class distribution (75% No DR). Focal Loss and SMOTE are compared later on the same files.

In [ ]:
# ===== X-Retina bootstrap: paste as the FIRST cell of every notebook =====
# Colab Secrets needed: GITHUB_TOKEN (or GITHUB-TOKEN). Optional: GIT_NAME, GIT_EMAIL, KAGGLE_USERNAME, KAGGLE_KEY
GITHUB_USER = "aimenkhan1"
REPO_NAME   = "X-Retina---Final-Year-Project-"
BRANCH      = "main"

import os, sys, json, subprocess
from google.colab import drive, userdata

def _secret(*names, default=None):
    for n in names:
        try:
            v = userdata.get(n)
            if v:
                return v
        except Exception:
            pass
    return default

GIT_NAME  = _secret("GIT_NAME",  default="X-Retina Team")
GIT_EMAIL = _secret("GIT_EMAIL", default="noreply@users.noreply.github.com")

# 1) Drive + shared folder
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/FYP(Gr2-Fall 2026)/XRetina_Shared"
assert os.path.isdir(DRIVE_ROOT), (
    f"Shared folder not found at {DRIVE_ROOT}. Add a shortcut named 'XRetina_Shared' to My Drive.")
for _sub in ("processed", "splits", "checkpoints", "outputs"):
    os.makedirs(f"{DRIVE_ROOT}/{_sub}", exist_ok=True)
os.environ["XRETINA_DRIVE_ROOT"] = DRIVE_ROOT
os.makedirs("/content/data/raw", exist_ok=True)
os.environ["XRETINA_RAW_DIR"] = "/content/data/raw"

# 2) Clone or update the repo (token is hidden from error messages)
_token = _secret("GITHUB_TOKEN", "GITHUB-TOKEN")
assert _token, "Add GITHUB_TOKEN (or GITHUB-TOKEN) in Colab Secrets and switch on Notebook access."
REPO_DIR = f"/content/{REPO_NAME}"
_url = f"https://{_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} failed:\n{r.stderr.replace(_token, '***')}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", _url, REPO_DIR])
_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", _url])
_run(["git", "-C", REPO_DIR, "fetch", "origin"])
try:
    _run(["git", "-C", REPO_DIR, "checkout", BRANCH])
except RuntimeError:
    _run(["git", "-C", REPO_DIR, "checkout", "-b", BRANCH])  # first time on a new branch
try:
    _run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])
except RuntimeError as e:
    print("Pull skipped (new local branch or local changes):", str(e).splitlines()[-1])
_run(["git", "-C", REPO_DIR, "config", "user.name", GIT_NAME])
_run(["git", "-C", REPO_DIR, "config", "user.email", GIT_EMAIL])

# 3) Requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

# 4) Kaggle credentials (only needed to download APTOS)
try:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    _kp = os.path.expanduser("~/.kaggle/kaggle.json")
    with open(_kp, "w") as f:
        json.dump({"username": userdata.get("KAGGLE_USERNAME"), "key": userdata.get("KAGGLE_KEY")}, f)
    os.chmod(_kp, 0o600)
except Exception as e:
    print("Kaggle secrets not set (only needed to download APTOS):", e)

# 5) Make `import src...` work
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

def git_push(message):
    """Commit and push your work directly to the current branch."""
    _run(["git", "-C", REPO_DIR, "add", "-A"])
    print(_run(["git", "-C", REPO_DIR, "commit", "-m", message]))
    print(_run(["git", "-C", REPO_DIR, "push", "-u", "origin", BRANCH]))

from src.config import get_paths
from src.utils import seed_everything, get_device
PATHS = get_paths()
print("Repo:", REPO_DIR, "| Branch:", BRANCH, "| Device:", get_device())
print({k: str(v) for k, v in PATHS.items()})

## 1. Settings
Only this cell is edited by hand. All preprocessing numbers are written down here and saved to `preprocessing_params.json`, so the report can quote them.

**Where the settings come from (checked in the PDFs)**
- 512 cache, CLAHE clip 2.0 / tile 8, crop threshold: **our own choices**. None of [7], [8], [9], [11] gives CLAHE values. [11] only says "contrast enhancement".
- [9] Patni: center crop + contrast normalisation, resize 224. [7], [8]: resize 224 + normalisation only.
- Split: [7] 85/15, [8] 80/10/10, [9] 80/20, all random image-level. We split **by patient** because EyePACS has two eyes per person.

In [ ]:
# ===== SETTINGS (edit only this cell) =====
MODE = "smoke"           # "smoke": ~300 images per dataset, writes NOTHING to Drive
                         # "full" : all images, writes splits + zips + manifest to Drive
SEED          = 42
OUT_SIZE      = 512      # cache size (train later at 380, or 224 for a like-for-like run with [11])
CLAHE_CLIP    = 2.0      # our setting (not from [11])
CLAHE_TILE    = 8
DARK_THR      = 10       # gray level below this counts as background when cropping
JPEG_QUALITY  = 95
SPLIT         = {"train": 0.70, "val": 0.15, "test": 0.15}
N_WORKERS     = 8
SMOKE_N       = 300
STRICT_COUNTS = True     # check the numbers found in notebook 01
FORCE_RESPLIT = False    # True only if the WHOLE team agrees to replace the frozen split
REPROCESS     = False    # True = ignore finished zips on Drive and process again

import time, json, zipfile, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
cv2.setNumThreads(1)

assert MODE in ("smoke", "full")
assert abs(sum(SPLIT.values()) - 1) < 1e-9
FULL = MODE == "full"
DRIVE     = Path(DRIVE_ROOT)
PROC_DIR  = DRIVE / "processed"
SPLIT_DIR = DRIVE / "splits"
OUT_DIR   = DRIVE / "outputs"
EYE_DIR   = Path("/content/drive/MyDrive/FYP(Gr2-Fall 2026)/Datasets/eyepacs/archive")
APTOS_DIR = Path(PATHS["raw"]) / "aptos2019"
LOCAL_OUT = Path("/content/processed_full" if FULL else "/content/processed_smoke")
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
META_DIR  = PROC_DIR if FULL else LOCAL_OUT          # where manifest/params go
SPLIT_FILE = SPLIT_DIR / "split_v1.csv"
PARAMS = dict(out_size=OUT_SIZE, clahe_clip=CLAHE_CLIP, clahe_tile=CLAHE_TILE,
              dark_threshold=DARK_THR, jpeg_quality=JPEG_QUALITY, split=SPLIT, seed=SEED,
              steps=["crop black border", "pad to square (black)", "resize (INTER_AREA)",
                     "CLAHE on L channel of LAB", "circular mask"],
              opencv=cv2.__version__)
print(f"MODE = {MODE}  ->", "writes to Drive" if FULL else "smoke test, nothing is written to Drive")

## 2. Load labels
EyePACS comes from your Drive copy. APTOS is downloaded to the local Colab disk (about 8 GB, needs the Kaggle secrets) only in `full` mode. In `smoke` mode APTOS is skipped unless it is already there.

In [ ]:
from src.data.download import download_dataset
from src.data.labels import load_labels, patient_id_from_name
from src.config import load_yaml
DS = load_yaml("datasets.yaml")

# A dataset is "finished" when its zip + meta are on Drive and the frozen split exists.
# Then the raw images are not needed again (labels are read back from the frozen split file).
def _finished(name):
    return (FULL and not REPROCESS and SPLIT_FILE.exists()
            and (PROC_DIR / f"xretina_{name}_{OUT_SIZE}.zip").exists()
            and (PROC_DIR / f"meta_{name}.csv").exists())

def _from_frozen(name):
    f = pd.read_csv(SPLIT_FILE, dtype={"image_id": str, "patient_id": str})
    f = f[f["dataset"] == name][["image_id", "label", "dataset", "patient_id"]].copy()
    f["eye"] = f["image_id"].str.rsplit("_", n=1).str[-1] if name == "eyepacs" else "unknown"
    f["image_path"] = ""      # raw not needed: this dataset is finished
    print(f"[{name}] finished earlier: labels read from the frozen split file (raw data not needed)")
    return f

# ---- EyePACS (Drive) ----
if not EYE_DIR.exists() and _finished("eyepacs"):
    eye_df = _from_frozen("eyepacs")
else:
    eye_df, _ = load_labels(EYE_DIR)
    eye_df = eye_df[["image_id", "label", "image_path"]].copy()
    eye_df["dataset"] = "eyepacs"
    _pid = eye_df["image_id"].map(patient_id_from_name)
    assert _pid.isna().sum() == 0, "some EyePACS images have no patient id"
    eye_df["patient_id"] = _pid.astype(str)
    eye_df["eye"] = eye_df["image_id"].astype(str).str.rsplit("_", n=1).str[-1]
print("EyePACS:", len(eye_df), "images |", eye_df["patient_id"].nunique(), "patients")
if STRICT_COUNTS:
    assert len(eye_df) == 16051, len(eye_df)
    assert eye_df["patient_id"].nunique() == 8205
    assert eye_df["label"].value_counts().sort_index().to_dict() == {0: 12050, 1: 1142, 2: 2223, 3: 358, 4: 278}

# ---- APTOS (local disk) ----
def _has_images(folder):
    if not folder.exists():
        return False
    return any(p.suffix.lower() in (".png", ".jpg", ".jpeg") for p in folder.rglob("*"))

frames = [eye_df]
ap_df = None
if _has_images(APTOS_DIR):
    ap_df, _ = load_labels(APTOS_DIR)
    ap_df = ap_df[["image_id", "label", "image_path"]].copy()
    ap_df["dataset"] = "aptos"
    ap_df["patient_id"] = ap_df["image_id"].astype(str)   # APTOS has no patient id: each image is its own group
    ap_df["eye"] = "unknown"
elif _finished("aptos"):
    ap_df = _from_frozen("aptos")
elif FULL:
    download_dataset(DS["aptos2019"]["kaggle_slug"], APTOS_DIR)   # about 8 GB, needs the Kaggle secrets
    ap_df, _ = load_labels(APTOS_DIR)
    ap_df = ap_df[["image_id", "label", "image_path"]].copy()
    ap_df["dataset"] = "aptos"
    ap_df["patient_id"] = ap_df["image_id"].astype(str)
    ap_df["eye"] = "unknown"
if ap_df is not None:
    print("APTOS:", len(ap_df), "images")
    if STRICT_COUNTS:
        assert len(ap_df) == 3662, len(ap_df)
        assert ap_df["label"].value_counts().sort_index().to_dict() == {0: 1805, 1: 370, 2: 999, 3: 193, 4: 295}
    frames.append(ap_df)
else:
    print("APTOS skipped (smoke test and not downloaded). The full run downloads it.")

all_df = pd.concat(frames, ignore_index=True)
all_df["image_id"] = all_df["image_id"].astype(str)
if not FULL:   # smoke test: small random sample per dataset
    all_df = pd.concat([g.sample(min(SMOKE_N, len(g)), random_state=SEED)
                        for _, g in all_df.groupby("dataset")], ignore_index=True)
print("\nImages used in this run:")
display(pd.crosstab(all_df["dataset"], all_df["label"], margins=True))

## 3. Preprocessing function
Steps, in this order:
1. **Crop** the black border around the retina
2. **Pad** to a square with black (so the retina is never stretched)
3. **Resize** to 512 x 512
4. **CLAHE** on the lightness channel only (colours stay natural)
5. **Circular mask** (black outside the circle)

The crop and padding numbers are saved in the manifest for every image. In FYP-II the IDRiD lesion masks must get exactly the same crop, so a lesion pixel at original position (x, y) lands at

`x_out = (x - crop_x0 + pad_left) * 512 / square_side` and `y_out = (y - crop_y0 + pad_top) * 512 / square_side`

In [ ]:
def preprocess_array(img):
    h0, w0 = img.shape[:2]
    # 1. crop the black border
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > DARK_THR
    rows = np.where(mask.mean(axis=1) > 0.01)[0]
    cols = np.where(mask.mean(axis=0) > 0.01)[0]
    if len(rows) == 0 or len(cols) == 0:          # blank image: keep everything
        y0, y1, x0, x1 = 0, h0, 0, w0
    else:
        y0, y1, x0, x1 = int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1
    img = img[y0:y1, x0:x1]
    # 2. pad to a square
    h, w = img.shape[:2]
    side = max(h, w)
    pt, pl = (side - h) // 2, (side - w) // 2
    img = cv2.copyMakeBorder(img, pt, side - h - pt, pl, side - w - pl,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    # 3. resize
    interp = cv2.INTER_AREA if side >= OUT_SIZE else cv2.INTER_CUBIC
    img = cv2.resize(img, (OUT_SIZE, OUT_SIZE), interpolation=interp)
    # 4. CLAHE on L channel
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(CLAHE_TILE, CLAHE_TILE))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    # 5. circular mask
    circ = np.zeros((OUT_SIZE, OUT_SIZE), np.uint8)
    cv2.circle(circ, (OUT_SIZE // 2, OUT_SIZE // 2), OUT_SIZE // 2, 255, -1)
    img = cv2.bitwise_and(img, img, mask=circ)
    meta = dict(orig_w=int(w0), orig_h=int(h0), crop_x0=x0, crop_y0=y0, crop_x1=x1, crop_y1=y1,
                pad_top=int(pt), pad_left=int(pl), square_side=int(side))
    return img, meta

def preprocess(path):
    img = cv2.imread(str(path))
    if img is None:
        raise IOError(f"cannot read {path}")
    return preprocess_array(img)

### Look at 5 examples (one per grade)
Check in the picture: the retina fills the circle, it is not stretched, vessels and spots look clearer but not noisy. If grade 3 or 4 images look over-bright, lower `CLAHE_CLIP` in the settings cell.

In [ ]:
demo_src = all_df[(all_df["dataset"] == "eyepacs") & (all_df["image_path"].astype(str) != "")]
if len(demo_src) == 0 or not Path(demo_src["image_path"].iloc[0]).exists():
    print("Raw EyePACS images are not available (dataset already finished): example figure skipped.")
else:
    demo = demo_src.groupby("label").sample(1, random_state=7).sort_values("label")
    fig, ax = plt.subplots(2, len(demo), figsize=(4 * len(demo), 8))
    t0 = time.time()
    for i, (_, r) in enumerate(demo.iterrows()):
        before = cv2.cvtColor(cv2.imread(str(r["image_path"])), cv2.COLOR_BGR2RGB)
        after, meta = preprocess(r["image_path"])
        ax[0, i].imshow(before); ax[0, i].set_title(f"Grade {r['label']} - before"); ax[0, i].axis("off")
        ax[1, i].imshow(cv2.cvtColor(after, cv2.COLOR_BGR2RGB)); ax[1, i].set_title("after"); ax[1, i].axis("off")
    per_img = (time.time() - t0) / len(demo)
    print(f"{per_img:.2f} s per image (one thread). Rough full-run estimate: "
          f"{per_img * 19713 / min(N_WORKERS, 4) / 60:.0f} min")
    plt.tight_layout()
    if FULL:
        plt.savefig(OUT_DIR / "preprocessing_examples.png", dpi=100)
    plt.show()

## 4. Patient-aware split (70 / 15 / 15)
- EyePACS: a patient has a left and a right eye. Both eyes always go to the **same** split, otherwise the model sees the same person in train and test.
- Each patient is stratified by their **worst eye grade**, so every split keeps the same grade mix.
- APTOS has no patient id, so each image is its own group (same as [7], [8], [9]). We say this in the report.
- Each dataset is split **separately**, so results can be reported per dataset later.
- The split is **frozen**: the first full run saves `splits/split_v1.csv`. Every later run (and every teammate) loads that file. All experiments (baseline, Focal Loss, SMOTE, hybrid) use the same split.

In [ ]:
def patient_level_split(df, seed):
    pat = df.groupby("patient_id")["label"].max().rename("worst").reset_index()
    def _tts(frame, size):
        try:
            return train_test_split(frame, test_size=size, stratify=frame["worst"], random_state=seed)
        except ValueError:
            print("  WARNING: a grade has too few patients to stratify -> plain random split (fine in smoke test only)")
            return train_test_split(frame, test_size=size, random_state=seed)
    rest_size = SPLIT["val"] + SPLIT["test"]
    tr, rest = _tts(pat, rest_size)
    va, te = _tts(rest, SPLIT["test"] / rest_size)
    m = {p: "train" for p in tr["patient_id"]}
    m.update({p: "val" for p in va["patient_id"]})
    m.update({p: "test" for p in te["patient_id"]})
    return df["patient_id"].map(m)

if FULL and SPLIT_FILE.exists() and not FORCE_RESPLIT:
    frozen = pd.read_csv(SPLIT_FILE, dtype={"image_id": str})
    all_df = all_df.merge(frozen[["dataset", "image_id", "split"]], on=["dataset", "image_id"], how="left")
    assert all_df["split"].notna().all(), "frozen split file does not cover all images"
    print("Loaded the FROZEN split:", SPLIT_FILE)
else:
    parts = []
    for name, g in all_df.groupby("dataset"):
        g = g.copy()
        g["split"] = patient_level_split(g, SEED)
        parts.append(g)
    all_df = pd.concat(parts, ignore_index=True)
    if FULL:
        SPLIT_DIR.mkdir(parents=True, exist_ok=True)
        all_df[["dataset", "image_id", "patient_id", "label", "split"]].to_csv(SPLIT_FILE, index=False)
        print("Saved NEW frozen split:", SPLIT_FILE)
    else:
        print("Smoke test: split not saved.")

# no patient may appear in two splits
for name, g in all_df.groupby("dataset"):
    n_split = g.groupby("patient_id")["split"].nunique()
    assert (n_split == 1).all(), f"{name}: a patient is in more than one split"
print("Leakage check passed: no patient is in two splits.\n")
for name, g in all_df.groupby("dataset"):
    print(name)
    tbl = pd.crosstab(g["split"], g["label"], margins=True).reindex(["train", "val", "test", "All"])
    display(tbl)
    print((g["split"].value_counts(normalize=True).reindex(["train", "val", "test"]) * 100).round(1).to_dict(), "(% of images)\n")

## 5. Process all images, then manifest + zip
- Output: 512 x 512 JPEG (quality 95). One zip per dataset, saved to `XRetina_Shared/processed/`.
- If Colab disconnects: run again. Finished datasets are skipped, finished images are kept (same session).
- Failed images are listed and left out of the manifest.

In [ ]:
def process_one(row):
    image_id, dataset, src, rel = row
    try:
        out, meta = preprocess(src)
        ok = cv2.imwrite(str(LOCAL_OUT / rel), out, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        if not ok:
            raise IOError("imwrite failed")
        return dict(image_id=image_id, dataset=dataset, rel_path=rel, ok=True, error="", **meta)
    except Exception as e:
        return dict(image_id=image_id, dataset=dataset, rel_path=rel, ok=False, error=str(e)[:200])

def run_dataset(name, chunk=1000):
    sub = all_df[all_df["dataset"] == name]
    zip_drive, meta_drive = PROC_DIR / f"xretina_{name}_{OUT_SIZE}.zip", PROC_DIR / f"meta_{name}.csv"
    if FULL and zip_drive.exists() and meta_drive.exists() and not REPROCESS:
        m = pd.read_csv(meta_drive, dtype={"image_id": str})
        if (m["ok"] == True).all():
            print(f"[{name}] already on Drive -> skipped (set REPROCESS = True to redo)")
            return m
        print(f"[{name}] an earlier run had failed images -> processing again")
    (LOCAL_OUT / name).mkdir(parents=True, exist_ok=True)
    ckpt = LOCAL_OUT / f"meta_{name}.csv"
    meta = pd.read_csv(ckpt, dtype={"image_id": str}) if ckpt.exists() else pd.DataFrame()
    done = set(meta.loc[meta["ok"] == True, "image_id"]) if len(meta) else set()
    rows = [(r.image_id, name, r.image_path, f"{name}/{r.image_id}.jpg")
            for r in sub.itertuples() if r.image_id not in done]
    print(f"[{name}] {len(sub)} images | {len(done)} already done | {len(rows)} to process")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for i in tqdm(range(0, len(rows), chunk), desc=name):
            res = list(ex.map(process_one, rows[i:i + chunk]))
            meta = pd.concat([meta, pd.DataFrame(res)], ignore_index=True)
            meta = meta.drop_duplicates("image_id", keep="last")
            meta.to_csv(ckpt, index=False)
    print(f"[{name}] processing took {(time.time() - t0) / 60:.1f} min")
    bad = meta[meta["ok"] != True]
    if len(bad):
        print(f"[{name}] FAILED images: {len(bad)}")
        display(bad.head(10))
    if FULL:
        good = meta[meta["ok"] == True]
        local_zip = LOCAL_OUT / zip_drive.name
        with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:
            for rel in tqdm(good["rel_path"], desc=f"zip {name}"):
                z.write(LOCAL_OUT / rel, arcname=rel)
        PROC_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_zip, zip_drive)
        assert zip_drive.stat().st_size == local_zip.stat().st_size, "zip copy to Drive is incomplete"
        meta.to_csv(meta_drive, index=False)
        print(f"[{name}] zip on Drive: {zip_drive.name} ({zip_drive.stat().st_size / 1e9:.2f} GB)")
    return meta

t_all = time.time()
meta_all = pd.concat([run_dataset(n) for n in all_df["dataset"].unique()], ignore_index=True)

manifest = all_df.drop(columns=["image_path"]).merge(
    meta_all, on=["dataset", "image_id"], how="left")
n_failed = int((manifest["ok"] != True).sum())
manifest_ok = manifest[manifest["ok"] == True].copy()
cols = ["image_id", "dataset", "patient_id", "eye", "label", "split", "rel_path",
        "orig_w", "orig_h", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "pad_top", "pad_left", "square_side"]
manifest_ok = manifest_ok[cols]
META_DIR.mkdir(parents=True, exist_ok=True)
manifest_ok.to_csv(META_DIR / "manifest.csv", index=False)
(META_DIR / "preprocessing_params.json").write_text(json.dumps(PARAMS, indent=2))
print(f"\nmanifest.csv: {len(manifest_ok)} rows | failed: {n_failed} | saved in {META_DIR}")
print(f"total time this session: {(time.time() - t_all) / 60:.1f} min")

## 6. Checks
All checks must pass before raw data is cleaned up.

In [ ]:
checks = {}
checks["no failed images"] = (n_failed == 0)
checks["manifest rows == images in this run"] = (len(manifest_ok) == len(all_df))

# every manifest file exists (inside the zip on Drive in full mode, on disk in smoke mode)
zips = {}
def read_processed(row_dataset, rel):
    if FULL:
        if row_dataset not in zips:
            zips[row_dataset] = zipfile.ZipFile(PROC_DIR / f"xretina_{row_dataset}_{OUT_SIZE}.zip")
        data = np.frombuffer(zips[row_dataset].read(rel), np.uint8)
        return cv2.imdecode(data, cv2.IMREAD_COLOR)
    return cv2.imread(str(LOCAL_OUT / rel))

if FULL:
    ok_files = True
    for name, g in manifest_ok.groupby("dataset"):
        z = zipfile.ZipFile(PROC_DIR / f"xretina_{name}_{OUT_SIZE}.zip")
        ok_files &= set(g["rel_path"]) <= set(z.namelist())
        ok_files &= (len(z.namelist()) == len(g))
    checks["every manifest file is in its zip (and nothing extra)"] = bool(ok_files)
else:
    checks["every manifest file is on disk"] = all((LOCAL_OUT / p).exists() for p in manifest_ok["rel_path"])

# no patient in two splits, from the manifest itself
leak = manifest_ok.groupby(["dataset", "patient_id"])["split"].nunique()
checks["no patient in two splits (manifest)"] = bool((leak == 1).all())

# crop sanity: how much of the original image was kept
manifest_ok["kept_frac"] = ((manifest_ok["crop_x1"] - manifest_ok["crop_x0"]) *
                            (manifest_ok["crop_y1"] - manifest_ok["crop_y0"])) / (manifest_ok["orig_w"] * manifest_ok["orig_h"])
odd = manifest_ok[manifest_ok["kept_frac"] < 0.25]
checks["very few odd crops (< 2% of images keep < 25% of the picture)"] = bool(len(odd) <= 0.02 * len(manifest_ok))

# pixel sanity on a random sample
smp = manifest_ok.sample(min(200, len(manifest_ok)), random_state=1)
shapes_ok, means = True, []
for r in smp.itertuples():
    im = read_processed(r.dataset, r.rel_path)
    shapes_ok &= (im is not None and im.shape == (OUT_SIZE, OUT_SIZE, 3))
    if im is not None:
        means.append(float(im.mean()))
checks[f"sample images are {OUT_SIZE}x{OUT_SIZE}x3"] = bool(shapes_ok)
checks["mean brightness is sensible (20-200)"] = bool(means) and 20 < np.mean(means) < 200

for k, v in checks.items():
    print("PASS" if v else "FAIL", "-", k)
CHECKS_PASSED = all(checks.values())
print("\nALL CHECKS PASSED" if CHECKS_PASSED else "\nSOME CHECKS FAILED - do not clean up yet")
print(f"Images with kept area < 25%: {len(odd)}")
if len(odd):
    display(odd[["dataset", "image_id", "label", "kept_frac"]].sort_values("kept_frac").head(8))

# visual check: 10 random processed images
show = manifest_ok.sample(min(10, len(manifest_ok)), random_state=3)
fig, ax = plt.subplots(2, 5, figsize=(20, 8))
for a, r in zip(ax.ravel(), show.itertuples()):
    a.imshow(cv2.cvtColor(read_processed(r.dataset, r.rel_path), cv2.COLOR_BGR2RGB))
    a.set_title(f"{r.dataset} | grade {r.label} | {r.split}"); a.axis("off")
plt.tight_layout(); plt.show()

# final tables + summary json
final_tbl = pd.crosstab([manifest_ok["dataset"], manifest_ok["split"]], manifest_ok["label"], margins=True)
display(final_tbl)
summary = dict(mode=MODE, params=PARAMS, n_images=int(len(manifest_ok)), n_failed=n_failed,
               checks={k: bool(v) for k, v in checks.items()},
               counts={f"{d}|{s}": int(n) for (d, s), n in manifest_ok.groupby(["dataset", "split"]).size().items()})
if FULL:
    (OUT_DIR / "preprocessing_summary.json").write_text(json.dumps(summary, indent=2))
    print("Saved:", OUT_DIR / "preprocessing_summary.json")

## 7. Raw cleanup (only after the checks pass, only in `full` mode)
- **APTOS local copy** (about 8 GB on the Colab disk): safe to delete, it can be downloaded again.
- **EyePACS on Drive**: this is your own 16,051-image subset of the original. **If it is deleted, this exact subset cannot be rebuilt.** The cell below only deletes it if you type the confirmation text. My advice: keep it until the mid-term (6-8 Nov) is finished.

In [ ]:
CLEAN_APTOS_LOCAL      = True
EYEPACS_DELETE_CONFIRM = ""      # to delete the Drive copy, type exactly: YES-DELETE-EYEPACS-RAW

if not FULL:
    print("Smoke mode: nothing to clean.")
elif not CHECKS_PASSED:
    print("Checks did not all pass: cleanup skipped.")
else:
    if CLEAN_APTOS_LOCAL and APTOS_DIR.exists():
        shutil.rmtree(APTOS_DIR)
        print("Deleted local APTOS raw:", APTOS_DIR)
    if EYEPACS_DELETE_CONFIRM == "YES-DELETE-EYEPACS-RAW":
        shutil.rmtree(EYE_DIR)
        print("Deleted EyePACS raw on Drive:", EYE_DIR)
    else:
        print("EyePACS raw on Drive was NOT deleted (no confirmation text).")
    print(f"Free local disk: {shutil.disk_usage('/content').free / 1e9:.1f} GB")

## For Maheen and Amna: load the processed data
Paste this in your notebook after the bootstrap cell. It copies nothing to Drive.

```python
import zipfile, pandas as pd
P = "/content/drive/MyDrive/FYP(Gr2-Fall 2026)/XRetina_Shared/processed"
for z in ("xretina_eyepacs_512.zip", "xretina_aptos_512.zip"):
    zipfile.ZipFile(f"{P}/{z}").extractall("/content/data/processed")
man = pd.read_csv(f"{P}/manifest.csv", dtype={"image_id": str})
man["path"] = "/content/data/processed/" + man["rel_path"]
train, val, test = (man[man.split == s] for s in ("train", "val", "test"))
```

Rules for everyone:
- Use the `split` column as it is. Never re-split.
- Do not oversample or augment inside these files. Augmentation, Focal Loss and SMOTE happen in the training code, on the **train** rows only.
- Images are 512 x 512 BGR-on-disk JPEG. Read with OpenCV or PIL, then resize to 380 (or 224 for the like-for-like run with [11]) in the dataloader.